# 02 · Tolles-Lawson compensation (interchangeable calibrators)
Compare calibration backends on the noisy cabin magnetometer `mag_4_uc`: built-in map-less and map-based modified. Metric: correlation with the map and residual std.

In [1]:
import os, sys
while not os.path.isdir('magnavlab') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('repo:', os.getcwd())

repo: /home/wpalka/magnav


In [2]:
import numpy as np
from magnavlab.io import load_flight, load_map, segment_indices
from magnavlab.calibration import BuiltinTL, MapBasedModifiedTL
nav = load_flight('data/Flt1003_train.h5')
mag_map = load_map('data/maps/Eastern_395.h5')
sl = segment_indices(nav, 50713.0, 54497.0)[::5]
lat = np.radians(nav.get('lat')[sl]); lon = np.radians(nav.get('lon')[sl])
# Earth's field (map-based target) at total-field scale = anomaly map + core
earth = mag_map.value(lat, lon) + (nav.get('mag_1_c')[sl] - nav.get('igrf')[sl])
flux = nav.flux(sl); z = nav.get('mag_4_uc')[sl]
half = sl.size // 2                      # paper protocol: 1st half = calibration

## Fit on the 1st half of the flight, evaluate on the 2nd half (validation)

In [3]:
val = slice(half, None)
def score(name, calibrator, target=None):
    tgt = None if target is None else target[:half]
    calibrator.fit(nav.flux(sl[:half]), z[:half], nav.dt, target=tgt)
    comp = calibrator.compensate(flux, z, nav.dt)
    corr = np.corrcoef(comp[val], earth[val])[0,1]
    print(f'{name:24s} corr with map={corr:.3f}  std(comp-map)={np.std(comp[val]-earth[val]):.0f} nT')

print('raw mag_4_uc'.ljust(24), f'corr with map={np.corrcoef(z[val], earth[val])[0,1]:.3f}  std={np.std(z[val]-earth[val]):.0f} nT')
score('BuiltinTL (map-less)', BuiltinTL())
score('MapBasedModifiedTL', MapBasedModifiedTL(), target=earth)

raw mag_4_uc             corr with map=0.830  std=207 nT
BuiltinTL (map-less)     corr with map=0.878  std(comp-map)=180 nT
MapBasedModifiedTL       corr with map=0.978  std(comp-map)=69 nT


The map-based modified variant gives the best fit to the map (in line with the paper), clearly better than the map-less approach on the validation half.